In this notebook I have created the bronze layer for this project. Created the catalog and schemas and made sure I loaded all the data to the tables.

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS permits;
CREATE SCHEMA IF NOT EXISTS permits.bronze;


In [0]:
from pyspark.sql.functions import lit, current_timestamp, col

base = "abfss://permits@permitsdata12.dfs.core.windows.net"
batch = "2026_09_15"

def load_bronze(folder, table):
    df = (spark.read
          .option("header", True)
          .option("inferSchema", False)
          .option("escape", '"')          # CSV standard uses "" to escape a quote, not backslash
          .option("multiLine", True)      # some DESCRIPTION values span multiple lines
          .csv(f"{base}/raw/{folder}")
          .drop("_id")
          .withColumn("_source_file", col("_metadata.file_path"))
          .withColumn("_loaded_at", current_timestamp())
          .withColumn("_batch_id", lit(batch)))

    df.write.mode("overwrite").saveAsTable(f"permits.bronze.{table}")
    print(f"{table}: {df.count():,} rows, {len(df.columns)} columns")

load_bronze("active", "active_permits")
load_bronze("cleared", "cleared_permits")

In [0]:
%sql
SELECT 'active' AS tbl, COUNT(*) AS rows FROM permits.bronze.active_permits
UNION ALL
SELECT 'cleared', COUNT(*) FROM permits.bronze.cleared_permits;

DESCRIBE permits.bronze.active_permits;

DESCRIBE permits.bronze.cleared_permits;

In [0]:
from pyspark.sql import functions as F

b = spark.table("permits.bronze.active_permits").unionByName(
    spark.table("permits.bronze.cleared_permits"))

print(f"{b.count():,} rows")   # expect 642,725

numeric_cols = ["DWELLING_UNITS_CREATED","DWELLING_UNITS_LOST","ASSEMBLY","INSTITUTIONAL",
                "RESIDENTIAL","BUSINESS_AND_PERSONAL_SERVICES","MERCANTILE","INDUSTRIAL",
                "INTERIOR_ALTERATIONS","DEMOLITION"]

for c in numeric_cols:
    n = b.filter(F.col(c).isNotNull() & ~F.trim(F.col(c)).rlike("^-?[0-9.]*$")).count()
    print(f"{c:32} {n:,}")

In [0]:
b.filter(F.col("RESIDENTIAL").isNotNull() & ~F.trim(F.col("RESIDENTIAL")).rlike("^-?[0-9.]*$")) \
 .select("PERMIT_NUM", "PERMIT_TYPE", "DESCRIPTION", "RESIDENTIAL") \
 .show(truncate=False)

In [0]:
from pyspark.sql import functions as F
b = spark.table("permits.bronze.active_permits").unionByName(
    spark.table("permits.bronze.cleared_permits"))

numeric_cols = [
    "DWELLING_UNITS_CREATED", "DWELLING_UNITS_LOST",
    "ASSEMBLY", "INSTITUTIONAL", "RESIDENTIAL",
    "BUSINESS_AND_PERSONAL_SERVICES", "MERCANTILE",
    "INDUSTRIAL", "INTERIOR_ALTERATIONS", "DEMOLITION",
]

for c in numeric_cols:
    bad = b.filter(F.col(c).isNotNull() & ~F.trim(F.col(c)).rlike("^-?[0-9.]*$"))
    n = bad.count()
    print(f"{c:32} {n:,}")
    if n:
        bad.select("PERMIT_NUM", "PERMIT_TYPE", c).show(3, truncate=False)